In [1]:
import torch
import torch.nn as nn

_ = torch.manual_seed(26)

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
hidden_size = 256
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=1
)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(
    dimension=0, size=sequence_len, step=sequence_len
)

In [4]:
# model declaration
class RecurrentNeuralNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embedding_size
        )
        self.input_linear = nn.Linear(
            in_features=embedding_size, out_features=hidden_size
        )
        self.hidden_linear = nn.Linear(
            in_features=hidden_size, out_features=hidden_size
        )
        self.output_linear = nn.Linear(in_features=hidden_size, out_features=vocab_size)

    def forward(self, x, h=None):
        batch_size = x.size(0)
        if h is None:
            h = torch.zeros(size=(batch_size, self.hidden_size))
        logits = []
        for i in range(x.size(1)):
            ch = x[:, i]
            new_h = torch.tanh(
                self.input_linear(self.embedding(ch))
                + self.hidden_linear(h)
            )
            logits.append(self.output_linear(new_h))
            h = new_h
        return torch.stack(tensors=logits, dim=1), h

In [5]:
# model initialization
model = RecurrentNeuralNetwork(
    vocab_size=vocab_size, embedding_size=embedding_size, hidden_size=hidden_size
)
print(f"parameters: {sum(param.numel() for param in model.parameters())}")
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

parameters: 93025


In [6]:
# training
for step in range(steps):
    optimizer.zero_grad()

    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    x_batch = x_train[batch]
    y_batch = y_train[batch]
    train_logits, _ = model(x_batch)
    train_loss = nn.functional.cross_entropy(
        input=train_logits.view(-1, vocab_size), target=y_batch.view(-1)
    )

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits, _ = model(x_val)
            val_loss = nn.functional.cross_entropy(
                input=val_logits.view(-1, vocab_size), target=y_val.view(-1)
            )
            print(
                f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}"
            )

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.2093892097473145, val loss = 4.212765693664551


Step 1000: train loss = 1.5857328176498413, val loss = 1.7054930925369263


Step 2000: train loss = 1.4672340154647827, val loss = 1.657588243484497


Step 3000: train loss = 1.4579397439956665, val loss = 1.6463452577590942


Step 4000: train loss = 1.4615370035171509, val loss = 1.6343305110931396


Step 4999: train loss = 1.3962292671203613, val loss = 1.6389050483703613


In [7]:
# prediction
with torch.inference_mode():
    output = ["R", "O", "M", "E", "O", ":", "\n"]
    max_output_length = 1000
    h = None

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)

    while len(output) < max_output_length:
        logits, h = model(i, h)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[output[-1]]]], dtype=torch.long)

In [8]:
print("".join(output))

ROMEO:
Before your dine!
If then, stand his flatterers
Thus more to end out of our capple, my open same, condems. Your thougreding, 'fail.
Then just make and weak to us others ages crowns.

LEONTES:
Not savily peace, and do you prettide that in upon my treaty,
Or's waons.
How to seen us of the grawn'd by my guiltof yet three of appines untinel woman's eyes,
How must not show'd of France with, for you
And up,
A putt he speak to with many country stand yield;
And, and my thereo--Pray, I will they Tyot?
The interped and forth for then of the sut you.
Rarried nor so hands being not, might have peet him as;
Farewell in tree. We'll not.

CORIOLANUS:
No free safeit's not thy potsured trencue your base in a signe.
Agoin much need Clauding holy
Derverson, sursh cannot stay
Makes and muty
Which now a peast as necessitation.

POMPEY:
Who field hangignify,
Having brough, none to like ready.
Hork'll bear York your earld,
And use me heaven and the age, when thou spop is thy men do,
He, in twines, to